In [6]:
# ============================================================
# Часть 0: Данные
# ============================================================

import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# %%
train_df = pd.read_csv("./data/in_domain_train.csv")
test_df = pd.read_csv("./data/in_domain_dev.csv")

print(train_df.head())
print(f"\nTrain size: {len(train_df)}, Test size: {len(test_df)}")
print(f"Label distribution (train):\n{train_df['acceptable'].value_counts()}")

# %%
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["sentence"].tolist(),
    train_df["acceptable"].tolist(),
    test_size=0.1,
    random_state=42,
    stratify=train_df["acceptable"],
)

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")

test_texts = test_df["sentence"].tolist()
test_labels = test_df["acceptable"].tolist()

# --- Веса классов для компенсации дисбаланса ---
# w_i = N / (C * n_i)  — обратно пропорционально частоте класса
# Идея: суммарный вклад каждого класса в loss одинаковый
n_total = len(train_labels)
n_pos = sum(train_labels)
n_neg = n_total - n_pos
n_classes = 2
class_weights = torch.tensor(
    [n_total / (n_classes * n_neg), n_total / (n_classes * n_pos)],
    dtype=torch.float32,
)
print(f"n_neg={n_neg}, n_pos={n_pos}")
print(f"Class weights: class0={class_weights[0]:.3f}, class1={class_weights[1]:.3f}")


   id                                           sentence  acceptable  \
0   0  Вдруг решетка беззвучно поехала в сторону, и н...           1   
1   1                       Этим летом не никуда ездили.           0   
2   2  Только Иван выразил какую бы то ни было готовн...           1   
3   3  Теперь ты видишь собственными глазами, как тут...           1   
4   4    На поверку вся теория оказалась полной чепухой.           1   

  error_type detailed_source  
0          0   Paducheva2004  
1     Syntax         Rusgram  
2          0   Paducheva2013  
3          0   Paducheva2010  
4          0   Paducheva2010  

Train size: 7869, Test size: 983
Label distribution (train):
acceptable
1    5864
0    2005
Name: count, dtype: int64
Train: 7082, Val: 787
n_neg=1804, n_pos=5278
Class weights: class0=1.963, class1=0.671


In [7]:
# ============================================================
# Часть 1: Fine-tuning RuBert
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

MODEL_NAME_RUBERT = "DeepPavlov/rubert-base-cased"

tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME_RUBERT)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


class RuCoLADataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


# %%
train_ds_bert = RuCoLADataset(train_texts, train_labels, tokenizer_bert)
val_ds_bert = RuCoLADataset(val_texts, val_labels, tokenizer_bert)
test_ds_bert = RuCoLADataset(test_texts, test_labels, tokenizer_bert)


def compute_metrics_classification(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "mcc": matthews_corrcoef(labels, preds),
    }


model_bert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_RUBERT, num_labels=2
)

training_args_bert = TrainingArguments(
    output_dir="./rubert_rucola",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="mcc",
    logging_steps=50,
)

trainer_bert = WeightedTrainer(
    model=model_bert,
    args=training_args_bert,
    train_dataset=train_ds_bert,
    eval_dataset=val_ds_bert,
    compute_metrics=compute_metrics_classification,
)

trainer_bert.train()

# %%
rubert_metrics = trainer_bert.evaluate(test_ds_bert)
print(f"[RuBert]  Accuracy: {rubert_metrics['eval_accuracy']:.4f}  "
      f"F1: {rubert_metrics['eval_f1']:.4f}  MCC: {rubert_metrics['eval_mcc']:.4f}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/666 [00:00<?, ?it/s]

{'loss': 0.6981, 'grad_norm': 1.9561786651611328, 'learning_rate': 1.84984984984985e-05, 'epoch': 0.23}
{'loss': 0.6581, 'grad_norm': 3.373408317565918, 'learning_rate': 1.6996996996997e-05, 'epoch': 0.45}
{'loss': 0.6741, 'grad_norm': 3.1062984466552734, 'learning_rate': 1.5495495495495498e-05, 'epoch': 0.68}
{'loss': 0.6442, 'grad_norm': 5.027284145355225, 'learning_rate': 1.3993993993993995e-05, 'epoch': 0.9}


  0%|          | 0/25 [00:00<?, ?it/s]

{'eval_loss': 0.6422727108001709, 'eval_accuracy': 0.6581956797966964, 'eval_f1': 0.7488328664799253, 'eval_mcc': 0.23888574791357509, 'eval_runtime': 0.9193, 'eval_samples_per_second': 856.067, 'eval_steps_per_second': 27.194, 'epoch': 1.0}
{'loss': 0.6276, 'grad_norm': 3.106212854385376, 'learning_rate': 1.2492492492492493e-05, 'epoch': 1.13}
{'loss': 0.5773, 'grad_norm': 3.918797016143799, 'learning_rate': 1.0990990990990992e-05, 'epoch': 1.35}
{'loss': 0.5884, 'grad_norm': 5.070089340209961, 'learning_rate': 9.489489489489491e-06, 'epoch': 1.58}
{'loss': 0.5489, 'grad_norm': 9.229273796081543, 'learning_rate': 7.987987987987988e-06, 'epoch': 1.8}


...

{'eval_loss': 0.6504855751991272, 'eval_accuracy': 0.761118170266836, 'eval_f1': 0.8481421647819063, 'eval_mcc': 0.30546732482826777, 'eval_runtime': 0.9182, 'eval_samples_per_second': 857.136, 'eval_steps_per_second': 27.228, 'epoch': 2.0}
{'loss': 0.5388, 'grad_norm': 8.814493179321289, 'learning_rate': 6.486486486486487e-06, 'epoch': 2.03}
{'loss': 0.4548, 'grad_norm': 13.9893798828125, 'learning_rate': 4.984984984984985e-06, 'epoch': 2.25}
{'loss': 0.4098, 'grad_norm': 11.999703407287598, 'learning_rate': 3.4834834834834835e-06, 'epoch': 2.48}
{'loss': 0.3999, 'grad_norm': 8.94623851776123, 'learning_rate': 1.9819819819819822e-06, 'epoch': 2.7}
{'loss': 0.4329, 'grad_norm': 15.504087448120117, 'learning_rate': 4.804804804804805e-07, 'epoch': 2.93}


  0%|          | 0/25 [00:00<?, ?it/s]

{'eval_loss': 0.7134513258934021, 'eval_accuracy': 0.770012706480305, 'eval_f1': 0.8505367464905037, 'eval_mcc': 0.3576286386217964, 'eval_runtime': 0.8955, 'eval_samples_per_second': 878.802, 'eval_steps_per_second': 27.916, 'epoch': 3.0}
{'train_runtime': 119.1968, 'train_samples_per_second': 178.243, 'train_steps_per_second': 5.587, 'train_loss': 0.5547013297095313, 'epoch': 3.0}


  0%|          | 0/31 [00:00<?, ?it/s]

[RuBert]  Accuracy: 0.7884  F1: 0.8644  MCC: 0.3952


In [ ]:
# ============================================================
# Часть 2: Zero-shot / Few-shot с RuGPT3
# ============================================================

from transformers import AutoModelForCausalLM

MODEL_NAME_GPT = "ai-forever/rugpt3large_based_on_gpt2"

tokenizer_gpt = AutoTokenizer.from_pretrained(MODEL_NAME_GPT)
model_gpt = AutoModelForCausalLM.from_pretrained(MODEL_NAME_GPT)
model_gpt.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gpt.to(device)


# --- Несколько вариантов промптов ---

PROMPT_VARIANTS = {
    "v1_direct": (
        "Является ли следующее предложение грамматически правильным?\n"
        "Предложение: {sentence}\nОтвет:"
    ),
    "v2_acceptable": (
        "Определи приемлемость предложения с точки зрения грамматики русского языка.\n"
        "Предложение: {sentence}\nОтвет (да/нет):"
    ),
    "v3_linguist": (
        "Ты — лингвист. Оцени грамматическую корректность предложения.\n"
        "Предложение: {sentence}\nГрамматически корректно (да/нет):"
    ),
}

YES_TOKEN_IDS = set(tokenizer_gpt.encode(" да", add_special_tokens=False) +
                     tokenizer_gpt.encode("да", add_special_tokens=False))
NO_TOKEN_IDS = set(tokenizer_gpt.encode(" нет", add_special_tokens=False) +
                    tokenizer_gpt.encode("нет", add_special_tokens=False))


def get_logit_diff(text, prompt_template, few_shot_examples=None):
    few_shot_examples = few_shot_examples or []

    prompt_parts = []
    for ex_text, ex_label in few_shot_examples:
        answer = "да" if ex_label == 1 else "нет"
        filled = prompt_template.format(sentence=ex_text) + " " + answer
        prompt_parts.append(filled)

    prompt_parts.append(prompt_template.format(sentence=text))
    full_prompt = "\n".join(prompt_parts)

    inputs = tokenizer_gpt(full_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_gpt(**inputs)

    next_token_logits = outputs.logits[0, -1, :]
    p_yes = next_token_logits[list(YES_TOKEN_IDS)].max().item()
    p_no = next_token_logits[list(NO_TOKEN_IDS)].max().item()

    return p_yes - p_no


def calibrate_threshold(texts, labels, prompt_template, few_shot_examples=None):
    diffs = [get_logit_diff(t, prompt_template, few_shot_examples) for t in tqdm(texts, desc="Calibrating")]
    best_thresh, best_mcc = 0, -1
    for thresh in np.linspace(min(diffs), max(diffs), 200):
        preds = [1 if d > thresh else 0 for d in diffs]
        mcc = matthews_corrcoef(labels, preds)
        if mcc > best_mcc:
            best_mcc = mcc
            best_thresh = thresh
    print(f"  Calibrated threshold={best_thresh:.4f}, val MCC={best_mcc:.4f}")
    return best_thresh


def classify_gpt(text, prompt_template, threshold=0.0, few_shot_examples=None):
    diff = get_logit_diff(text, prompt_template, few_shot_examples)
    return 1 if diff > threshold else 0


# --- Выбор few-shot примеров из тренировки ---
few_shot_pool = list(zip(train_texts, train_labels))


def pick_few_shot(n):
    if n == 0:
        return []
    pos = [x for x in few_shot_pool if x[1] == 1]
    neg = [x for x in few_shot_pool if x[1] == 0]
    half = n // 2
    rest = n - half
    return pos[:half] + neg[:rest]


# --- 2а: Перебор промптов (few_shot=2) ---

print("\n=== 2а: Перебор промптов (few-shot=2) ===")
few2 = pick_few_shot(2)
gpt_prompt_results = {}

for pname, ptemplate in PROMPT_VARIANTS.items():
    thresh = calibrate_threshold(val_texts, val_labels, ptemplate, few_shot_examples=few2)
    preds = [classify_gpt(t, ptemplate, threshold=thresh, few_shot_examples=few2)
             for t in tqdm(test_texts, desc=f"Prompt {pname}")]
    acc = accuracy_score(test_labels, preds)
    f1 = f1_score(test_labels, preds)
    mcc = matthews_corrcoef(test_labels, preds)
    gpt_prompt_results[pname] = {"accuracy": acc, "f1": f1, "mcc": mcc}
    print(f"  [{pname}]  Acc: {acc:.4f}  F1: {f1:.4f}  MCC: {mcc:.4f}")

best_prompt = max(gpt_prompt_results, key=lambda k: gpt_prompt_results[k]["mcc"])
print(f"\nЛучший промпт: {best_prompt}")

# --- 2б: Разное число few-shot (0, 1, 2, 4) ---

print("\n=== 2б: Разное число few-shot ===")
gpt_fewshot_results = {}
best_template = PROMPT_VARIANTS[best_prompt]

for n_shot in [0, 1, 2, 4]:
    examples = pick_few_shot(n_shot)
    thresh = calibrate_threshold(val_texts, val_labels, best_template, few_shot_examples=examples)
    preds = [classify_gpt(t, best_template, threshold=thresh, few_shot_examples=examples)
             for t in tqdm(test_texts, desc=f"Few-shot={n_shot}")]
    acc = accuracy_score(test_labels, preds)
    f1 = f1_score(test_labels, preds)
    mcc = matthews_corrcoef(test_labels, preds)
    gpt_fewshot_results[n_shot] = {"accuracy": acc, "f1": f1, "mcc": mcc}
    print(f"  [few-shot={n_shot}]  Acc: {acc:.4f}  F1: {f1:.4f}  MCC: {mcc:.4f}")

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 24f81c23-a196-41b1-a020-0f0757a352d3)')' thrown while requesting HEAD https://huggingface.co/ai-forever/rugpt3large_based_on_gpt2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].



=== 2а: Перебор промптов (few-shot=2) ===


Calibrating: 100%|██████████| 787/787 [02:21<00:00,  5.55it/s]


  Calibrated threshold=-0.8067, val MCC=0.0278


Prompt v1_direct:  24%|██▍       | 239/983 [00:43<02:15,  5.49it/s]

In [15]:
# ============================================================
# Часть 3: Fine-tuning RuT5
# ============================================================

from transformers import T5ForConditionalGeneration, T5Tokenizer

MODEL_NAME_T5 = "cointegrated/rut5-base"

tokenizer_t5 = T5Tokenizer.from_pretrained(MODEL_NAME_T5)
model_t5 = T5ForConditionalGeneration.from_pretrained(MODEL_NAME_T5)


LABEL_POS = "да"
LABEL_NEG = "нет"


class T5RuCoLADataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.input_texts = [f"Определи грамматическую корректность: {t}" for t in texts]
        self.target_texts = [LABEL_POS if l == 1 else LABEL_NEG for l in labels]

        self.inputs = tokenizer(
            self.input_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
        )
        self.targets = tokenizer(
            self.target_texts,
            padding=True,
            truncation=True,
            max_length=8,
        )

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.inputs["input_ids"][idx]),
            "attention_mask": torch.tensor(self.inputs["attention_mask"][idx]),
            "labels": torch.tensor(self.targets["input_ids"][idx]),
        }

    def __len__(self):
        return len(self.input_texts)


train_ds_t5 = T5RuCoLADataset(train_texts, train_labels, tokenizer_t5)
val_ds_t5 = T5RuCoLADataset(val_texts, val_labels, tokenizer_t5)

# --- Oversampling миноритного класса для компенсации дисбаланса ---
pos_idx = [i for i, l in enumerate(train_labels) if l == 1]
neg_idx = [i for i, l in enumerate(train_labels) if l == 0]
oversample_factor = len(pos_idx) // len(neg_idx)
neg_oversampled = neg_idx * oversample_factor
balanced_idx = pos_idx + neg_oversampled
np.random.shuffle(balanced_idx)

train_texts_balanced = [train_texts[i] for i in balanced_idx]
train_labels_balanced = [train_labels[i] for i in balanced_idx]

train_ds_t5 = T5RuCoLADataset(train_texts_balanced, train_labels_balanced, tokenizer_t5)


def compute_metrics_t5(eval_pred):
    predictions, labels = eval_pred
    pred_ids = np.argmax(predictions, axis=-1)
    pred_texts = tokenizer_t5.batch_decode(pred_ids, skip_special_tokens=True)
    label_texts = tokenizer_t5.batch_decode(labels, skip_special_tokens=True)

    pred_texts = [p.strip().lower() for p in pred_texts]
    label_texts = [l.strip().lower() for l in label_texts]

    pred_binary = [1 if p == LABEL_POS else 0 for p in pred_texts]
    label_binary = [1 if l == LABEL_POS else 0 for l in label_texts]

    return {
        "accuracy": accuracy_score(label_binary, pred_binary),
        "f1": f1_score(label_binary, pred_binary),
        "mcc": matthews_corrcoef(label_binary, pred_binary),
    }


training_args_t5 = TrainingArguments(
    output_dir="./rut5_rucola",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mcc",
    predict_with_generate=True,
    generation_max_length=8,
    logging_steps=50,
)

trainer_t5 = Trainer(
    model=model_t5,
    args=training_args_t5,
    train_dataset=train_ds_t5,
    eval_dataset=val_ds_t5,
    compute_metrics=compute_metrics_t5,
)

trainer_t5.train()

# --- Оценка на тесте ---
test_ds_t5 = T5RuCoLADataset(test_texts, test_labels, tokenizer_t5)
rut5_metrics = trainer_t5.evaluate(test_ds_t5)
print(f"[RuT5]  Accuracy: {rut5_metrics['eval_accuracy']:.4f}  "
      f"F1: {rut5_metrics['eval_f1']:.4f}  MCC: {rut5_metrics['eval_mcc']:.4f}")

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'predict_with_generate'